In [1]:
import json
from openai import OpenAI

# Connect to LM Studio local server
client = OpenAI(
    base_url="http://10.0.0.159:1234/v1",
    api_key="lm-studio"
)

In [ ]:
def smart_home_control(json_file, command):

    # Load JSON
    with open(json_file, "r") as f:
        data = json.load(f)

    json_text = json.dumps(data, indent=2)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a smart home controller.\n"
                "You MUST ONLY use devices that exist in the provided JSON.\n"
                "You are NOT allowed to assume devices exist.\n"
                "You are NOT allowed to invent devices.\n"
            )
        },
        {
            "role": "user",
            "content": f"""
    Command: {command}

    Devices:
    {json_text}

    Return format:

    If success:
    {{
    "status": "success",
    "devices": {{ ... }},
    "explanation": "short explanation here"
    }}

    If failure:
    {{
    "status": "failure",
    "explanation": "short explanation why it failed"
    }}
    """
        }
    ]

    response = client.chat.completions.create(
        model="meta-llama-3.1-8b-instruct",   # LM Studio uses this alias
        messages=messages,
        temperature=0.3,
        max_tokens=50
    )

    return response.choices[0].message.content.strip()




In [6]:
if __name__ == "__main__":
    user_command = input("Enter command: ")
    response = smart_home_control("data/home1.json", user_command)
    print(response)

{
  "status": "success",
  "devices": {
    "kitchen": {
      "overhead_light": {
        "type": "light",
        "on": true,
        "brightness": 0
      }
    }
  },
  "explanation": "Turned on the kitchen overhead light"
}
